In [3]:
import joblib
import numpy as np

# Load trained components
iso = joblib.load("iforest_model.pkl")
scaler = joblib.load("feature_scaler.pkl")

# Example chosen threshold (replace with your real tuned value)
THRESHOLD = 0.56  

# -----------------------------
# 1️⃣ Extract model structure
# -----------------------------
n_trees = len(iso.estimators_)
print(f"Isolation Forest Trees: {n_trees}")

# Gather all tree thresholds and features
all_thresholds = []
all_features = []
for tree in iso.estimators_:
    tree_struct = tree.tree_
    all_thresholds.append(tree_struct.threshold)
    all_features.append(tree_struct.feature)

# Flatten arrays
thresholds_flat = np.concatenate(all_thresholds)
features_flat = np.concatenate(all_features)

# -----------------------------
# 2️⃣ Q15 Fixed-Point Quantization
# -----------------------------
def to_q15(x):
    return np.int16(np.clip(x * 32768, -32768, 32767))

thresholds_q15 = to_q15(thresholds_flat / np.max(np.abs(thresholds_flat)))
scaler_mean_q15 = to_q15(scaler.mean_ / np.max(np.abs(scaler.mean_)))
scaler_scale_q15 = to_q15(scaler.scale_ / np.max(np.abs(scaler.scale_)))
threshold_q15 = to_q15(THRESHOLD / np.max(np.abs(THRESHOLD)))

# -----------------------------
# 3️⃣ Generate header content safely (no f-string concat errors)
# -----------------------------
header_lines = [
    "/* =============================================== */",
    "/*   EclipseGuardian: SEL Detector (Q15 Model)     */",
    "/*   Auto-generated from Python model              */",
    "/* =============================================== */",
    "#ifndef MODEL_IFOREST_H",
    "#define MODEL_IFOREST_H",
    "",
    f"#define NUM_TREES {n_trees}",
    f"#define NUM_FEATURES {len(scaler.mean_)}",
    f"#define MODEL_THRESHOLD_Q15 {int(threshold_q15)}",
    "",
    "static const int16_t scaler_mean_q15[NUM_FEATURES] = {" +
    ", ".join(map(str, scaler_mean_q15.tolist())) + "};",
    "",
    "static const int16_t scaler_scale_q15[NUM_FEATURES] = {" +
    ", ".join(map(str, scaler_scale_q15.tolist())) + "};",
    "",
    f"static const int16_t thresholds_q15[{len(thresholds_q15)}] = " +
    "{" + ", ".join(map(str, thresholds_q15.tolist())) + "};",
    "",
    f"static const int8_t features_idx[{len(features_flat)}] = " +
    "{" + ", ".join(map(str, features_flat.tolist())) + "};",
    "",
    "#endif // MODEL_IFOREST_H"
]

with open("model_iforest.h", "w") as f:
    f.write("\n".join(header_lines))

print("✅ model_iforest.h successfully generated!")


Isolation Forest Trees: 100
✅ model_iforest.h successfully generated!
